# Trading AI - Walk Forward Fine-Tuning
## Fine-tune Qwen 2.5 1.5B on Indian Market Data

Steps:
1. Install dependencies
2. Upload dataset zip
3. Fine-tune with LoRA
4. Evaluate on test set
5. Save and download model


In [ ]:
# Check GPU
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU')
print('CUDA:', torch.version.cuda)
print('Memory:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')


In [ ]:
# Install dependencies
!pip install -q unsloth
!pip install -q transformers datasets
!pip install -q trl peft accelerate
!pip install -q bitsandbytes
print('Dependencies installed')


In [ ]:
# Upload your dataset zip file
from google.colab import files
import zipfile
import os

print('Please upload your window zip file')
uploaded = files.upload()

# Extract zip
for fname in uploaded.keys():
    print(f'Extracting {fname}...')
    with zipfile.ZipFile(fname, 'r') as z:
        z.extractall('/content/dataset')

# List files
print(os.listdir('/content/dataset'))


In [ ]:
# Load training dataset
import json

def load_jsonl(path):
    data = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data

train_data = load_jsonl('/content/dataset/train.jsonl')
test_data  = load_jsonl('/content/dataset/test.jsonl')

print(f'Train samples: {len(train_data)}')
print(f'Test samples : {len(test_data)}')

# Preview
sample = train_data[0]
print('Sample prompt (first 500 chars):')
print(sample['instruction'][:500])
print('Sample output:')
print(sample['output'][:300])


In [ ]:
# Load Qwen 2.5 1.5B with Unsloth
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048
DTYPE          = None
LOAD_IN_4BIT   = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = 'unsloth/Qwen2.5-1.5B-Instruct',
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = DTYPE,
    load_in_4bit   = LOAD_IN_4BIT,
)

print('Model loaded successfully')
total = sum(p.numel() for p in model.parameters())
print(f'Model params: {total/1e6:.0f}M')


In [ ]:
# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = [
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj'
    ],
    lora_alpha     = 16,
    lora_dropout   = 0,
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state   = 42,
    use_rslora     = False,
    loftq_config   = None,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable/1e6:.1f}M / {total/1e6:.0f}M ({trainable/total*100:.1f}%)')


In [ ]:
# Format dataset for training
from datasets import Dataset

SYSTEM_PROMPT = (
    'You are a professional trading '
    'intelligence system specializing '
    'in Indian markets. Analyze market '
    'data and provide structured '
    'JSON trading decisions.'
)

def format_sample(sample):
    messages = [
        {'role': 'system',    'content': SYSTEM_PROMPT},
        {'role': 'user',      'content': sample['instruction']},
        {'role': 'assistant', 'content': sample['output']}
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize              = False,
        add_generation_prompt = False
    )

train_texts   = [format_sample(s) for s in train_data]
train_dataset = Dataset.from_dict({'text': train_texts})

print(f'Dataset size: {len(train_dataset)}')
print('Sample (first 200 chars):')
print(train_dataset[0]['text'][:200])


In [ ]:
# Fine-tune with SFTTrainer
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = train_dataset,
    dataset_text_field = 'text',
    max_seq_length     = MAX_SEQ_LENGTH,
    dataset_num_proc   = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps                = 5,
        num_train_epochs            = 3,
        learning_rate               = 2e-4,
        fp16        = not is_bfloat16_supported(),
        bf16        = is_bfloat16_supported(),
        logging_steps               = 10,
        optim                       = 'adamw_8bit',
        weight_decay                = 0.01,
        lr_scheduler_type           = 'linear',
        seed                        = 42,
        output_dir                  = '/content/outputs',
    ),
)

gpu_stats = torch.cuda.get_device_properties(0)
max_memory = round(gpu_stats.total_memory / 1024**3, 1)
print(f'GPU: {gpu_stats.name}')
print(f'Max memory: {max_memory} GB')
print('Starting training...')

trainer_stats = trainer.train()

print('Training complete!')
print(f'Training loss: {trainer_stats.training_loss:.4f}')


In [ ]:
# Evaluate on test set
import json
from collections import Counter

FastLanguageModel.for_inference(model)

def predict(instruction):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': instruction}
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize              = True,
        add_generation_prompt = True,
        return_tensors        = 'pt'
    ).to('cuda')

    with torch.no_grad():
        outputs = model.generate(
            input_ids      = inputs,
            max_new_tokens = 512,
            temperature    = 0.1,
            do_sample      = True,
        )

    response = tokenizer.decode(
        outputs[0][inputs.shape[1]:],
        skip_special_tokens = True
    )
    return response

# Evaluate on first 50 samples
eval_samples = test_data[:50]
action_match = 0
valid_json   = 0
results      = []

print('Evaluating on 50 test samples...')

for i, sample in enumerate(eval_samples):
    try:
        response = predict(sample['instruction'])

        if '```' in response:
            response = response.split('```')[1]
            if response.startswith('json'):
                response = response[4:]

        pred   = json.loads(response)
        actual = json.loads(sample['output'])
        valid_json += 1

        if pred.get('action') == actual.get('action'):
            action_match += 1

        results.append({
            'predicted': pred.get('action'),
            'actual'   : actual.get('action'),
            'match'    : pred.get('action') == actual.get('action')
        })

        if (i + 1) % 10 == 0:
            print(f'  Processed {i+1}/50...')

    except Exception as e:
        results.append({'error': str(e)})

print(f'Valid JSON   : {valid_json}/50')
print(f'Action Match : {action_match}/50 ({action_match/50*100:.0f}%)')

pred_actions = Counter(r.get('predicted') for r in results if r.get('predicted'))
print(f'Predicted Actions: {dict(pred_actions)}')


In [ ]:
# Save fine-tuned model
import os

SAVE_PATH = '/content/trading_ai_model'

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f'Model saved to {SAVE_PATH}')
for f in os.listdir(SAVE_PATH):
    size = os.path.getsize(os.path.join(SAVE_PATH, f)) / 1024**2
    print(f'  {f}: {size:.1f} MB')


In [ ]:
# Save as GGUF for Ollama
GGUF_PATH = '/content/trading_ai_gguf'

model.save_pretrained_gguf(
    GGUF_PATH,
    tokenizer,
    quantization_method = 'q4_k_m'
)

print('GGUF model saved!')
for f in os.listdir(GGUF_PATH):
    size = os.path.getsize(os.path.join(GGUF_PATH, f)) / 1024**2
    print(f'  {f}: {size:.1f} MB')


In [ ]:
# Download model to your PC
import shutil
from google.colab import files

# Zip LoRA weights
print('Zipping LoRA weights...')
shutil.make_archive(
    '/content/trading_ai_lora',
    'zip',
    '/content/trading_ai_model'
)

# Download LoRA
print('Downloading LoRA weights...')
files.download('/content/trading_ai_lora.zip')

# Download GGUF
import os
gguf_files = [
    f for f in os.listdir('/content/trading_ai_gguf')
    if f.endswith('.gguf')
]
for f in gguf_files:
    print(f'Downloading {f}...')
    files.download(f'/content/trading_ai_gguf/{f}')

print('Download complete!')
